# Audio Dubbing / Cut Detector (final)

**Task:** detect, per 30s social-media segment, whether the audio contains (a) a **cut**: non-AI audio spliced in from another source, and (b) **AI dubbing**: part or all of the clip is AI-generated voice. The four classes (`real`, `cut`, `dubbed`, `cut_and_dubbed`) are derived from those two binary outputs. `music` clips are excluded.

**What this notebook does**
1. Syncs `.npz` filenames from Kaggle into the labeling Google Sheet (labels preserved).
2. Loads real `(80, 3001)` log-Mel `.npz` files into RAM (no synthetic fallback; missing data raises).
3. Audits sheet labels against the per-segment tags embedded in the filenames, and builds grouping keys from the filenames so sibling segments of one video never straddle train and validation.
4. Engineered input channels, computed on the GPU from the pad-aware normalized log-Mel: `[static, delta, splice-step]`.
5. Two-head ResNet-18 (max + mean pooling), BCE loss, square-root smoothed sampler, GPU-side augmentation.
6. Stratified **grouped** 5-fold CV with out-of-fold probabilities, per-head threshold tuning, derived 4-class report.
7. Final model, checkpoint export, inference pipeline, preprocessing parity check, external/codec robustness test.
8. Optional: WavLM embedding probe from the raw MP4 dataset to compare against the CNN.

The filename is used **only** for grouping and the label audit. It never reaches the model.

## 1. Setup and Imports

In [ ]:
!pip install --upgrade kaggle --quiet

In [ ]:
import os
import io
import sys
import json
import builtins
import zipfile
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models

import gspread
from google.colab import auth, files
from google.auth import default

# Patch exit to handle script terminations gracefully in interactive kernels
builtins.exit = sys.exit

print("Core libraries successfully imported.")
print(f"PyTorch version: {torch.__version__}")

Core libraries successfully imported.
PyTorch version: 2.14.1+cu130


In [ ]:
import re, itertools, collections, subprocess
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (average_precision_score, precision_recall_curve, f1_score,
                             classification_report, confusion_matrix)
from tqdm.auto import tqdm
import librosa
print("Extra imports ready.")

Extra imports ready.


## 2. Authentication & API Setup

In [ ]:
# 1. Authenticate Google User for Google Sheets access
print("Authenticating Google account...")
auth.authenticate_user()
creds, _ = default()
gs_client = gspread.authorize(creds)
print("Google Sheets API client successfully authorized.\n")

# 2. Configure ~/.kaggle/ directory and place credentials BEFORE importing KaggleApi
kaggle_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_dir, exist_ok=True)
target_path = os.path.join(kaggle_dir, 'kaggle.json')

# Clean up stale/empty environment variables if present
for env_var in ['KAGGLE_USERNAME', 'KAGGLE_KEY']:
    if env_var in os.environ and not os.environ[env_var]:
        del os.environ[env_var]

if not os.path.exists(target_path):
    if os.path.exists('kaggle.json'):
        with open('kaggle.json', 'r') as src, open(target_path, 'w') as dst:
            dst.write(src.read())
        print("Moved kaggle.json from working directory to ~/.kaggle/.")
    else:
        print("Kaggle credentials not detected. Please upload 'kaggle.json':")
        uploaded = files.upload()
        if 'kaggle.json' in uploaded:
            with open(target_path, 'wb') as f:
                f.write(uploaded['kaggle.json'])
            print("kaggle.json successfully uploaded.")

if os.path.exists(target_path):
    os.chmod(target_path, 0o600)
else:
    raise FileNotFoundError("kaggle.json was not found. Please re-run Cell 2 and upload the file.")

# 3. Import and authenticate KaggleApi ONLY AFTER kaggle.json exists on disk
from kaggle.api.kaggle_api_extended import KaggleApi

api = KaggleApi()
api.authenticate()
print("Kaggle API successfully authenticated!")

Authenticating Google account...
Google Sheets API client successfully authorized.

Kaggle credentials not detected. Please upload 'kaggle.json':


Saving kaggle.json to kaggle.json
kaggle.json successfully uploaded.
Kaggle API successfully authenticated!


## 3. Sync Kaggle Dataset Files with Google Sheets

In [ ]:
import os
import time
import requests

SPREADSHEET_ID = '1zI5Zj_G64kvpQk-ZMS6TzfNQPj9XNyP960X-lkjWcIM'
DATASET_NAME = 'dherylc/spectrogram-ai-social-media-short-video-detector'

# 1. Access Google Sheet
try:
    sh = gs_client.open_by_key(SPREADSHEET_ID)
    worksheet = sh.get_worksheet(0)
    print(f"Connected to Google Spreadsheet: '{sh.title}'")
except Exception as e:
    print(f"Error connecting to spreadsheet: {e}")
    raise e

# 2. Read existing records and normalize to base filenames
records = worksheet.get_all_records()
existing_normalized = set()

for r in records:
    fname = r.get('filename') or r.get('file_name')
    if fname:
        base_fname = os.path.basename(str(fname).strip())
        if not base_fname.lower().endswith('.npz'):
            base_fname += '.npz'
        existing_normalized.add(base_fname)

print(f"Retrieved {len(existing_normalized)} unique normalized filenames from Google Sheet.")

# 3. Query all Kaggle Dataset files handling API pagination with rate-limit protection
kaggle_npz_files = []
dataset_slug = DATASET_NAME.split('/')[-1]
local_dataset_path = f"/kaggle/input/{dataset_slug}"

# Method A: Direct local filesystem check (If running inside Kaggle Notebook)
if os.path.exists(local_dataset_path):
    print(f"Reading dataset directly from local filesystem: {local_dataset_path}...")
    for root, _, fnames in os.walk(local_dataset_path):
        for fn in fnames:
            if fn.endswith('.npz'):
                kaggle_npz_files.append(fn)
    print(f"Found {len(kaggle_npz_files)} total .npz files from local filesystem.")

# Method B: Paginated REST API call with token iteration
else:
    try:
        from kaggle.api.kaggle_api_extended import KaggleApi
        api = KaggleApi()
        api.authenticate()

        config = getattr(api, 'config', {})
        if isinstance(config, dict):
            user_val = config.get('username') or config.get('user')
            auth_val = config.get('key') or config.get('apiKey')
        else:
            user_val = getattr(config, 'username', None)
            auth_val = getattr(config, 'key', None)

        if not user_val or not auth_val:
            user_val = os.environ.get('KAGGLE_USERNAME')
            auth_val = os.environ.get('KAGGLE_KEY')

        owner, dataset = DATASET_NAME.split('/')
        url = f"https://www.kaggle.com/api/v1/datasets/list/{owner}/{dataset}"

        print("Fetching full file listing via Kaggle API with token pagination...")

        all_dataset_files = []
        page_token = None

        while True:
            params = {'pageSize': 1000}
            if page_token:
                params['pageToken'] = page_token

            # Exponential delay retry for 429 rate limit safety
            for attempt in range(3):
                response = requests.get(url, auth=(user_val, auth_val), params=params)
                if response.status_code == 429:
                    print("Rate limit reached. Pausing 3 seconds before retry...")
                    time.sleep(3)
                    continue
                break

            response.raise_for_status()
            data = response.json()

            page_files = data.get('datasetFiles', []) or data.get('files', [])
            if not page_files:
                break

            all_dataset_files.extend(page_files)

            # Retrieve nextPageToken from response
            page_token = data.get('nextPageToken') or data.get('next_page_token')
            if not page_token or len(page_files) == 0:
                break

        kaggle_npz_files = [
            os.path.basename(f.get('name', ''))
            for f in all_dataset_files
            if f.get('name', '').endswith('.npz')
        ]
        print(f"Found {len(kaggle_npz_files)} total .npz files from Kaggle dataset.")
    except Exception as e:
        print(f"Notice: Unable to query Kaggle dataset ({e}). Using existing sheet records.")

# 4. Identify and append missing files safely without duplicates
new_rows = []
for fname in kaggle_npz_files:
    normalized_fname = fname.strip()
    if not normalized_fname.lower().endswith('.npz'):
        normalized_fname += '.npz'
    if normalized_fname not in existing_normalized:
        new_rows.append([normalized_fname, 'Unclassified'])
        existing_normalized.add(normalized_fname)

if new_rows:
    print(f"Appending {len(new_rows)} uniquely missing files to Google Sheet...")
    # Chunk inserts into batches of 500 to prevent Google Sheets API payload caps
    batch_size = 500
    for i in range(0, len(new_rows), batch_size):
        worksheet.append_rows(new_rows[i:i + batch_size])
    print(f"Successfully appended {len(new_rows)} new files. Synchronization complete.")
else:
    print("All dataset files are already synchronized. No duplicates appended.")

Connected to Google Spreadsheet: 'content_classification'
Retrieved 1596 unique normalized filenames from Google Sheet.
Fetching full file listing via Kaggle API with token pagination...
Found 1596 total .npz files from Kaggle dataset.
All dataset files are already synchronized. No duplicates appended.


## 4. Configuration

Everything tunable lives here. `TIME_STEPS` is derived from the clip length so training and inference cannot drift apart.

In [ ]:
SEED = 42
SR = 16000
HOP = 160
CLIP_SECONDS = 30
NUM_MELS = 80
TIME_STEPS = CLIP_SECONDS * SR // HOP + 1          # 3001 frames for 30s at hop 160

EXCLUDE_LABELS = {'music'}
CLASSES = ['cut', 'cut_and_dubbed', 'dubbed', 'real']   # music excluded
HEAD_NAMES = ['cut', 'dub']                              # output 0 = contains splice, output 1 = contains AI voice

EPOCHS = 6              # val metrics plateaued around epoch 5-6 in the first run
BATCH_SIZE = 32
N_FOLDS = 5
RUN_CV = True
TRAIN_FINAL = True
PRETRAINED = True

# Engineered input channels: [static, delta, splice-step]
#   win          : half-window (frames, 10ms each) for the splice-step channel
#   static_norm  : 'bin' = per-frequency-bin z-score over active frames, 'clip' = one z-score per clip
FEATURE_CFG = dict(win=25, static_norm='bin')

AUDIT_ACTION = 'report'     # 'report' = only print label/filename disagreements, 'drop' = also remove them from training
RAW_MP4_DATASET = None      # optional: Kaggle slug of the raw MP4 dataset for the embedding probe, e.g. "user/dataset-name"

THRESHOLDS = {'cut': 0.5, 'dub': 0.5}   # overwritten by tuned values after CV / loaded from checkpoint

np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
use_cuda = device.type == 'cuda'
print(f"device={device} | frames per clip={TIME_STEPS} | classes={CLASSES}")

device=cuda | frames per clip=3001 | classes=['cut', 'cut_and_dubbed', 'dubbed', 'real']


## 5. Data: Index, Label Audit, Groups, In-Memory Cache

**Filename convention (inferred from the sheet):** `<handle><video id><tags>_NNN.npz`. `_001`, `_002`, ... are consecutive 30s segments of one video, and `<tags>` appears to list the per-segment labels in order (`cutdubreal` = segment 1 cut+dub, segment 2 real). The audit below tests that hypothesis on your whole sheet and prints the agreement rate. Disagreements are either label errors or a wrong reading of the tags, so check them by hand.

**Groups:** the filename with `_NNN.npz` removed. All segments of a video share it, and CV never splits a group.

In [ ]:
import os

# ---- 1. locate (or download) the .npz files
kaggle_local = f"/kaggle/input/{DATASET_NAME.split('/')[-1]}"
DATA_DIR = kaggle_local if os.path.isdir(kaggle_local) else "/content/npz_features"

def _scan(d):
    return {f: os.path.join(r, f) for r, _, fs in os.walk(d) for f in fs if f.endswith('.npz')}

NPZ_INDEX = _scan(DATA_DIR) if os.path.isdir(DATA_DIR) else {}
if not NPZ_INDEX:
    os.makedirs(DATA_DIR, exist_ok=True)
    print(f"Downloading {DATASET_NAME} to {DATA_DIR} ...")
    api.dataset_download_files(DATASET_NAME, path=DATA_DIR, unzip=True)
    NPZ_INDEX = _scan(DATA_DIR)
print(f"Indexed {len(NPZ_INDEX)} .npz files under {DATA_DIR}")

# ---- 2. read labels from the sheet
worksheet = sh.get_worksheet(0)
rows = []
for r in worksheet.get_all_records():
    fname = r.get('filename') or r.get('file_name')
    label = str(r.get('manual_classification') or r.get('classification') or 'unclassified').strip().lower()
    if fname and label != 'unclassified':
        fname = os.path.basename(str(fname).strip())
        if not fname.lower().endswith('.npz'):
            fname += '.npz'
        rows.append({'filename': fname, 'label': label})
if not rows:
    raise RuntimeError("No labeled rows in the sheet. Label data before training.")

# ---- 3. filename parsing: group key + tag audit
TOKEN_TO_LABEL = {'cutdub': 'cut_and_dubbed', 'cut': 'cut', 'dub': 'dubbed', 'real': 'real', 'music': 'music'}
NAME_RE = re.compile(r'^(?P<pre>.*?)(?P<vid>\d{15,20})(?P<tags>[a-z]+)_(?P<seg>\d+)\.npz$', re.IGNORECASE)

def group_key(fname):
    return re.sub(r'_\d+\.npz$', '', fname, flags=re.IGNORECASE)

def seg_index(fname):
    m = re.search(r'_(\d+)\.npz$', fname, flags=re.IGNORECASE)
    return int(m.group(1)) if m else 1

def _tokenizations(s):
    if not s:
        return [[]]
    out = []
    for tok in TOKEN_TO_LABEL:
        if s.startswith(tok):
            out += [[tok] + rest for rest in _tokenizations(s[len(tok):])]
    return out

def audit_record(fname, label, base=1):
    m = NAME_RE.match(fname)
    if not m or len(m.group('tags')) > 80:
        return 'unparseable'
    toks = [t for t in _tokenizations(m.group('tags').lower()) if t]
    if not toks:
        return 'unparseable'
    k = int(m.group('seg')) - base
    ok = any(0 <= k < len(t) and TOKEN_TO_LABEL[t[k]] == label for t in toks)
    return 'match' if ok else 'mismatch'

for base in (1, 0):
    c = collections.Counter(audit_record(r['filename'], r['label'], base) for r in rows)
    agree = c['match'] / max(c['match'] + c['mismatch'], 1)
    print(f"[audit, segment numbering starts at {base}] {dict(c)} -> agreement {agree:.1%}")

for r in rows:
    r['audit'] = audit_record(r['filename'], r['label'], base=1)
bad = pd.DataFrame([r for r in rows if r['audit'] == 'mismatch'])[['filename', 'label']] if any(r['audit'] == 'mismatch' for r in rows) else pd.DataFrame()
if len(bad):
    print(f"\n{len(bad)} rows whose sheet label disagrees with the filename tag (review these by hand):")
    print(bad.head(25).to_string(index=False))
if AUDIT_ACTION == 'drop':
    rows = [r for r in rows if r['audit'] != 'mismatch']
    print(f"AUDIT_ACTION='drop': removed {len(bad)} disagreeing rows")

# ---- 4. drop excluded classes, validate
n_before = len(rows)
labeled_records = [r for r in rows if r['label'] not in EXCLUDE_LABELS]
print(f"\nExcluded {n_before - len(labeled_records)} rows with labels {sorted(EXCLUDE_LABELS)}")
found = sorted(set(r['label'] for r in labeled_records))
assert found == sorted(CLASSES), f"Unexpected label set: {found}"
missing = [r['filename'] for r in labeled_records if r['filename'] not in NPZ_INDEX]
assert not missing, f"{len(missing)} labeled files not on disk, e.g. {missing[:3]}"

N = len(labeled_records)
GROUPS = np.array([group_key(r['filename']) for r in labeled_records])
SEG = np.array([seg_index(r['filename']) for r in labeled_records])
Y_CLS = np.array([CLASSES.index(r['label']) for r in labeled_records])
CLS_TO_HEADS = {'real': (0, 0), 'cut': (1, 0), 'dubbed': (0, 1), 'cut_and_dubbed': (1, 1)}
Y_HEADS = np.array([CLS_TO_HEADS[r['label']] for r in labeled_records], dtype=np.float32)
_LUT = np.array([[CLASSES.index('real'), CLASSES.index('dubbed')],
                 [CLASSES.index('cut'), CLASSES.index('cut_and_dubbed')]])

print(f"N={N} | videos(groups)={len(set(GROUPS))} | max segments per video={max(collections.Counter(GROUPS).values())}")
print("Class counts:", dict(zip(CLASSES, np.bincount(Y_CLS, minlength=len(CLASSES)))))
print(f"Positives: cut={int(Y_HEADS[:, 0].sum())}, dub={int(Y_HEADS[:, 1].sum())}")

# ---- 5. pad-aware normalization (single implementation, shared with inference) and RAM cache
def pad_aware_normalize(spec, eps=1e-8):
    """Min-max scale using ONLY active (non-zero-padded) columns; padding stays exactly 0."""
    active = np.sum(np.abs(spec), axis=0) > 0
    out = np.zeros_like(spec)
    if np.any(active):
        a = spec[:, active]
        out[:, active] = (a - a.min()) / (a.max() - a.min() + eps)
    return out

def load_spec(path):
    d = np.load(path)
    a = d[d.files[0]].astype(np.float32)
    assert a.shape[0] == NUM_MELS, f"{path}: expected {NUM_MELS} mel bins, got {a.shape}"
    a = a[:, :TIME_STEPS]
    if a.shape[1] < TIME_STEPS:
        a = np.pad(a, ((0, 0), (0, TIME_STEPS - a.shape[1])))
    return pad_aware_normalize(a)

_cache = np.zeros((N, NUM_MELS, TIME_STEPS), dtype=np.float32)
for i, r in enumerate(tqdm(labeled_records, desc="Caching spectrograms")):
    _cache[i] = load_spec(NPZ_INDEX[r['filename']])
SPEC_CACHE = torch.from_numpy(_cache)
print(f"SPEC_CACHE {tuple(SPEC_CACHE.shape)} | {SPEC_CACHE.numel() * 4 / 1e9:.2f} GB")

Dataset URL: https://www.kaggle.com/datasets/dherylc/spectrogram-ai-social-media-short-video-detector
Indexed 1596 .npz files under /content/npz_features
[audit, segment numbering starts at 1] {'match': 416, 'mismatch': 51, 'unparseable': 1129} -> agreement 89.1%
[audit, segment numbering starts at 0] {'mismatch': 327, 'match': 140, 'unparseable': 1129} -> agreement 30.0%

51 rows whose sheet label disagrees with the filename tag (review these by hand):
                                                                                      filename          label
               asmrsuaranafaskarniilyas1jutakaliasmrrelaxmeme7570601665926286599cutcut_003.npz            cut
                                           yntkts1jutakalifyp7564401007556381960cutcut_003.npz            cut
                                                    clipviralfyp7686462606500039954cut_002.npz            cut
      BagaimanabisaanggaranBPOMlebihkecildaripengadaankauskakiDi7633715500043799829cut_001.npz cut_and

Caching spectrograms:   0%|          | 0/1591 [00:00<?, ?it/s]

SPEC_CACHE (1591, 80, 3001) | 1.53 GB


### Shortcut check: clip length vs class

Clips shorter than 30s carry a zero-padded tail, so the active length is visible to the model. If it differs strongly by class, the network can learn duration instead of manipulation artifacts, which will not transfer to arbitrary uploads.

In [ ]:
active_len = np.zeros(N, dtype=int)
for s in range(0, N, 128):
    active_len[s:s + 128] = (SPEC_CACHE[s:s + 128].abs().sum(1) > 0).sum(1).numpy()
print("Active frames per class [p5, median, p95]:")
for k, name in enumerate(CLASSES):
    print(f"  {name:16s} n={int((Y_CLS == k).sum()):4d}", np.percentile(active_len[Y_CLS == k], [5, 50, 95]).astype(int))
print("Fraction of clips shorter than the full window, per class:")
for k, name in enumerate(CLASSES):
    print(f"  {name:16s} {np.mean(active_len[Y_CLS == k] < TIME_STEPS - 5):.2f}")

Active frames per class [p5, median, p95]:
  cut              n=  84 [ 349 2757 3001]
  cut_and_dubbed   n= 104 [ 799 2941 2987]
  dubbed           n= 149 [ 607 2279 2996]
  real             n=1254 [1354 3001 3001]
Fraction of clips shorter than the full window, per class:
  cut              0.88
  cut_and_dubbed   0.98
  dubbed           0.95
  real             0.14


## 6. Feature Engineering (GPU, on the fly)

Input is the pad-aware normalized log-Mel `(B, 1, 80, T)`. Output is `(B, 3, 80, T)`:

| Channel | What it captures |
|---|---|
| 0 static | Per-bin z-score over active frames. Keeps relative spectral structure and removes the clip-wide level. |
| 1 delta | Frame-to-frame change (transients, onsets), z-scored. |
| 2 splice-step | Mean spectrum of the next `win` frames minus the previous `win` frames. A splice from another source shifts noise floor and room tone for a sustained stretch, which shows up as a step across many bins. Frames within `win` of the end of the audio are zeroed to avoid fake boundary events. |

All statistics are computed over active frames only, and padding stays exactly 0 in every channel.

Augmentation (training only): Gaussian noise on active frames, plus one frequency mask. No time masking, because it can erase the 1-2s edit that carries the label.

In [ ]:
def _masked_zscore(c, m, per_bin=False, eps=1e-5):
    """c: (B,1,F,T) values, m: (B,1,1,T) float mask of valid frames. Padding/invalid frames come back as 0."""
    if per_bin:
        n = m.sum(-1, keepdim=True).clamp(min=1)
        mu = (c * m).sum(-1, keepdim=True) / n
        var = (((c - mu) * m) ** 2).sum(-1, keepdim=True) / n
    else:
        n = (m.sum(dim=(2, 3), keepdim=True) * c.shape[2]).clamp(min=1)
        mu = (c * m).sum(dim=(2, 3), keepdim=True) / n
        var = (((c - mu) * m) ** 2).sum(dim=(2, 3), keepdim=True) / n
    return (c - mu) / torch.sqrt(var + eps) * m

def build_features(x, win=25, static_norm='bin'):
    """(B,1,F,T) normalized log-Mel -> (B,3,F,T) [static, delta, splice-step]."""
    x = x.float()
    B, _, Fq, T = x.shape
    active = x.abs().sum(dim=(1, 2)) > 0                    # (B,T)
    am = active[:, None, None, :].to(x.dtype)                # (B,1,1,T)
    L = active.sum(1)                                        # (B,) active frame count

    static = _masked_zscore(x, am, per_bin=(static_norm == 'bin'))

    delta = torch.diff(x, dim=-1, prepend=x[..., :1]) * am
    delta = _masked_zscore(delta, am, per_bin=False)

    xp = F.pad(x, (win, win - 1, 0, 0), mode='replicate')    # edge-replicate in time
    avg = F.avg_pool2d(xp, kernel_size=(1, win), stride=1)   # (B,1,F,T+win): window means
    step = avg[..., win:win + T] - avg[..., :T]              # mean(next win frames) - mean(previous win frames)
    valid = (torch.arange(T, device=x.device)[None, :] < (L[:, None] - win)).to(x.dtype)[:, None, None, :]
    step = _masked_zscore(step * valid, valid, per_bin=False)

    return torch.cat([static, delta, step], dim=1)

def augment_batch(x, p_noise=0.5, p_mask=0.5, noise_std=0.02, max_w=10):
    """GPU augmentation on normalized log-Mel (B,1,F,T). Noise only on active frames; one frequency mask per sample."""
    B, _, Fq, T = x.shape
    dev = x.device
    active = (x.abs().sum(dim=(1, 2)) > 0)[:, None, None, :].to(x.dtype)
    use_noise = (torch.rand(B, 1, 1, 1, device=dev) < p_noise).to(x.dtype)
    x = x + use_noise * noise_std * torch.randn_like(x) * active
    w = torch.randint(1, max_w + 1, (B,), device=dev)
    f0 = (torch.rand(B, device=dev) * (Fq - w)).long()
    ar = torch.arange(Fq, device=dev)[None, :]
    fm = (ar >= f0[:, None]) & (ar < (f0 + w)[:, None]) & (torch.rand(B, 1, device=dev) < p_mask)
    return x.masked_fill(fm[:, None, :, None], 0)

# self-check: shapes, finiteness, padding stays exactly zero in all channels
_t = torch.rand(2, 1, NUM_MELS, TIME_STEPS)
_t[1, :, :, TIME_STEPS // 2:] = 0
_f = build_features(_t, **FEATURE_CFG)
assert _f.shape == (2, 3, NUM_MELS, TIME_STEPS) and torch.isfinite(_f).all()
assert _f[1, :, :, TIME_STEPS // 2:].abs().max().item() == 0.0
assert torch.isfinite(augment_batch(_t)).all()
print("Feature builder self-check passed:", tuple(_f.shape))

Feature builder self-check passed: (2, 3, 80, 3001)


## 7. Model, Sampler and Training Functions

Two-head ResNet-18 (ImageNet weights): max pooling (peak anomaly for short edits) and mean pooling (for clips that are dubbed throughout) are concatenated before the head. Loss is plain `BCEWithLogitsLoss`; class balance comes from the square-root smoothed sampler only (weighted loss plus sampler over-corrected earlier).

In [ ]:
class TwoHeadResNet(nn.Module):
    def __init__(self, n_out=2, pretrained=True):
        super().__init__()
        r = models.resnet18(weights=models.ResNet18_Weights.DEFAULT if pretrained else None)
        self.backbone = nn.Sequential(r.conv1, r.bn1, r.relu, r.maxpool, r.layer1, r.layer2, r.layer3, r.layer4)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Sequential(nn.Linear(1024, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.5), nn.Linear(256, n_out))

    def forward(self, x):
        f = self.backbone(x)
        return self.head(torch.cat([self.max_pool(f).flatten(1), self.avg_pool(f).flatten(1)], dim=1))

class SpecDataset(Dataset):
    def __init__(self, idx):
        self.idx = np.asarray(idx)
    def __len__(self):
        return len(self.idx)
    def __getitem__(self, i):
        j = int(self.idx[i])
        return SPEC_CACHE[j].unsqueeze(0), torch.from_numpy(Y_HEADS[j])

def make_train_loader(train_idx):
    cls = Y_CLS[train_idx]
    counts = np.bincount(cls, minlength=len(CLASSES))
    w = 1.0 / np.sqrt(np.maximum(counts, 1))               # square-root smoothed inverse frequency
    sampler = WeightedRandomSampler(torch.tensor(w[cls], dtype=torch.double), num_samples=len(train_idx),
                                    replacement=True, generator=torch.Generator().manual_seed(SEED))
    return DataLoader(SpecDataset(train_idx), batch_size=BATCH_SIZE, sampler=sampler,
                      drop_last=len(train_idx) > BATCH_SIZE, num_workers=0, pin_memory=use_cuda)

@torch.no_grad()
def predict_indices(model, idx, batch_size=64):
    """Sigmoid probabilities (len(idx), 2) for cached samples."""
    model.eval()
    out = []
    for x, _ in DataLoader(SpecDataset(idx), batch_size=batch_size, shuffle=False, num_workers=0):
        out.append(torch.sigmoid(model(build_features(x.to(device), **FEATURE_CFG))).cpu().numpy())
    return np.concatenate(out)

def _safe_ap(y, p):
    return average_precision_score(y, p) if 0 < y.sum() < len(y) else float('nan')

def train_model(train_idx, val_idx=None, epochs=EPOCHS):
    model = TwoHeadResNet(pretrained=PRETRAINED).to(device)
    loader = make_train_loader(np.asarray(train_idx))
    optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=2e-3)
    scheduler = optim.lr_scheduler.OneCycleLR(optimizer, max_lr=3e-4, steps_per_epoch=len(loader), epochs=epochs, pct_start=0.2)
    scaler = torch.amp.GradScaler("cuda", enabled=use_cuda)
    criterion = nn.BCEWithLogitsLoss()

    for ep in range(1, epochs + 1):
        model.train()
        total, seen = 0.0, 0
        for x, y in tqdm(loader, desc=f"epoch {ep}/{epochs}", leave=False):
            x, y = x.to(device, non_blocking=True), y.to(device)
            with torch.no_grad():
                feats = build_features(augment_batch(x), **FEATURE_CFG)
            optimizer.zero_grad()
            with torch.amp.autocast(device_type="cuda", enabled=use_cuda):
                logits = model(feats)
            loss = criterion(logits.float(), y)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            total += loss.item() * x.size(0)
            seen += x.size(0)
        msg = f"  epoch {ep:02d}/{epochs} train loss {total / max(seen, 1):.4f}"
        if val_idx is not None:
            p = predict_indices(model, val_idx)
            msg += f" | val AP cut {_safe_ap(Y_HEADS[val_idx, 0], p[:, 0]):.3f} dub {_safe_ap(Y_HEADS[val_idx, 1], p[:, 1]):.3f}"
        print(msg)
    return model

def probs_to_class(p, thr=None):
    thr = thr or THRESHOLDS
    return _LUT[(p[:, 0] >= thr['cut']).astype(int), (p[:, 1] >= thr['dub']).astype(int)]
print("Training utilities ready.")

Training utilities ready.


## 8. Grouped Cross-Validation

`StratifiedGroupKFold` keeps every video's segments in one fold and balances the four classes across folds. Each fold trains from scratch for a fixed number of epochs, so the out-of-fold probabilities are an honest estimate with no selection on the held-out fold.

In [ ]:
sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
FOLDS = list(sgkf.split(np.zeros(N), Y_CLS, GROUPS))
OOF = np.full((N, 2), np.nan, dtype=np.float32)

if RUN_CV:
    for k, (tr, va) in enumerate(FOLDS):
        assert not (set(GROUPS[tr]) & set(GROUPS[va])), "group leakage between train and validation"
        print(f"\n=== Fold {k + 1}/{N_FOLDS} | train {len(tr)} | val {len(va)} | val classes {dict(zip(CLASSES, np.bincount(Y_CLS[va], minlength=len(CLASSES))))}")
        m = train_model(tr, va)
        OOF[va] = predict_indices(m, va)
        del m
        if use_cuda:
            torch.cuda.empty_cache()
    assert np.isfinite(OOF).all()
    np.save("oof_probs.npy", OOF)
    print("\nSaved oof_probs.npy")
else:
    print("RUN_CV=False: skipped.")


=== Fold 1/5 | train 1270 | val 321 | val classes {'cut': np.int64(18), 'cut_and_dubbed': np.int64(18), 'dubbed': np.int64(29), 'real': np.int64(256)}
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 188MB/s]


epoch 1/6:   0%|          | 0/39 [00:00<?, ?it/s]

  epoch 01/6 train loss 0.6149 | val AP cut 0.416 dub 0.824


epoch 2/6:   0%|          | 0/39 [00:00<?, ?it/s]

  epoch 02/6 train loss 0.3209 | val AP cut 0.490 dub 0.761


... (52 outputs left out)

epoch 5/6:   0%|          | 0/42 [00:00<?, ?it/s]

  epoch 05/6 train loss 0.0790 | val AP cut 0.544 dub 0.894


epoch 6/6:   0%|          | 0/42 [00:00<?, ?it/s]

  epoch 06/6 train loss 0.0782 | val AP cut 0.547 dub 0.890

Saved oof_probs.npy


### Out-of-fold report and threshold tuning

Per-head average precision (threshold-free), then F1 at 0.5 and at the F1-optimal threshold. The tuned thresholds are chosen on the same out-of-fold predictions they are scored on, so tuned scores are slightly optimistic. Headline operating numbers: false-alarm rate on `real` and recall on manipulated clips.

In [ ]:
def best_threshold(y, p):
    if not (0 < y.sum() < len(y)):
        return 0.5, float('nan')
    prec, rec, thr = precision_recall_curve(y, p)
    f1 = (2 * prec * rec / np.maximum(prec + rec, 1e-9))[:-1]
    i = int(np.nanargmax(f1))
    return float(thr[i]), float(f1[i])

if RUN_CV and np.isfinite(OOF).all():
    print(f"{'head':5s} {'AP':>6s} {'F1@0.5':>8s} {'thr*':>6s} {'F1@thr*':>8s}")
    for h, name in enumerate(HEAD_NAMES):
        y, p = Y_HEADS[:, h], OOF[:, h]
        t, f = best_threshold(y, p)
        THRESHOLDS[name] = t
        print(f"{name:5s} {_safe_ap(y, p):6.3f} {f1_score(y, p >= 0.5):8.3f} {t:6.2f} {f:8.3f}")

    for title, thr in (("threshold 0.5", {'cut': 0.5, 'dub': 0.5}), ("tuned thresholds", THRESHOLDS)):
        pred = probs_to_class(OOF, thr)
        print(f"\n--- derived 4-class, {title}: macro-F1 {f1_score(Y_CLS, pred, average='macro'):.3f}")
    pred = probs_to_class(OOF)
    print("\n" + classification_report(Y_CLS, pred, target_names=CLASSES, zero_division=0))
    print("Confusion matrix (rows=true, cols=pred):", CLASSES)
    print(confusion_matrix(Y_CLS, pred, labels=range(len(CLASSES))))

    flagged = pred != CLASSES.index('real')
    real_mask = Y_CLS == CLASSES.index('real')
    print(f"\nFalse-alarm rate on real clips: {flagged[real_mask].mean():.3f}")
    print(f"Recall on manipulated clips (any manipulation flagged): {flagged[~real_mask].mean():.3f}")
    print(f"AP for 'any manipulation' (max of heads): {_safe_ap((~real_mask).astype(int), OOF.max(1)):.3f}")
    print("Tuned thresholds:", {k: round(v, 3) for k, v in THRESHOLDS.items()})
else:
    print("No OOF predictions available; thresholds stay at defaults.")

head      AP   F1@0.5   thr*  F1@thr*
cut    0.507    0.480   0.34    0.557
dub    0.852    0.808   0.43    0.822

--- derived 4-class, threshold 0.5: macro-F1 0.579

--- derived 4-class, tuned thresholds: macro-F1 0.616

                precision    recall  f1-score   support

           cut       0.32      0.50      0.39        84
cut_and_dubbed       0.46      0.65      0.54       104
        dubbed       0.64      0.56      0.60       149
          real       0.96      0.91      0.93      1254

      accuracy                           0.84      1591
     macro avg       0.60      0.66      0.62      1591
  weighted avg       0.87      0.84      0.85      1591

Confusion matrix (rows=true, cols=pred): ['cut', 'cut_and_dubbed', 'dubbed', 'real']
[[  42   13    8   21]
 [   7   68   21    8]
 [   5   46   84   14]
 [  76   22   18 1138]]

False-alarm rate on real clips: 0.093
Recall on manipulated clips (any manipulation flagged): 0.872
AP for 'any manipulation' (max of heads): 0.839


## 9. Final Model and Checkpoint

Trains on all labeled data with the same fixed epoch count, then exports weights together with class order, thresholds and the exact preprocessing/feature config. Run this after the CV report looks acceptable.

In [ ]:
try:
    import google.colab.files as gfiles
except ImportError:
    gfiles = None

CKPT = "audio_deepfake_resnet18_weights.pt"
PREPROCESS = dict(sr=SR, n_fft=400, win_length=400, hop_length=HOP, n_mels=NUM_MELS, frames=TIME_STEPS,
                  fmin=0, fmax=8000, power=2.0, ref="max", top_db=80, norm="pad_aware_minmax",
                  padding="spectrogram_tail_zero")

def load_model(path, dev=None):
    dev = dev or device
    try:
        ck = torch.load(path, map_location=dev)
    except Exception:
        ck = torch.load(path, map_location=dev, weights_only=False)
    m = TwoHeadResNet(pretrained=False)
    m.load_state_dict(ck["state_dict"])
    FEATURE_CFG.update(ck["feature_cfg"])
    THRESHOLDS.update(ck["thresholds"])
    return m.to(dev).eval()

if TRAIN_FINAL:
    final_model = train_model(np.arange(N), None)
    torch.save({"state_dict": final_model.state_dict(), "classes": CLASSES, "heads": HEAD_NAMES,
                "thresholds": {k: float(v) for k, v in THRESHOLDS.items()},
                "feature_cfg": dict(FEATURE_CFG), "preprocess": PREPROCESS}, CKPT)
    print(f"Saved {CKPT} ({os.path.getsize(CKPT) / 1e6:.1f} MB) with thresholds {THRESHOLDS}")
    if gfiles is not None:
        gfiles.download(CKPT)
    model = final_model
else:
    print("TRAIN_FINAL=False: skipped.")

epoch 1/6:   0%|          | 0/49 [00:00<?, ?it/s]

  epoch 01/6 train loss 0.5711


epoch 2/6:   0%|          | 0/49 [00:00<?, ?it/s]

  epoch 02/6 train loss 0.2871


epoch 3/6:   0%|          | 0/49 [00:00<?, ?it/s]

  epoch 03/6 train loss 0.1629


... (4 outputs left out)

epoch 6/6:   0%|          | 0/49 [00:00<?, ?it/s]

  epoch 06/6 train loss 0.0687
Saved audio_deepfake_resnet18_weights.pt (45.8 MB) with thresholds {'cut': 0.33616000413894653, 'dub': 0.42712879180908203}


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 10. Inference Pipeline

Mirrors training exactly: mel spectrogram of the raw 30s chunk, tail zero-padding **in the spectrogram domain** (as in the training `.npz` files), the shared pad-aware normalization, then the same GPU feature builder. A trailing chunk shorter than 5s is dropped when the file has more than one chunk.

Per-segment flags use the tuned thresholds. Music-only chunks were excluded from training, so their output is not meaningful.

In [ ]:
def mel_tensor(audio_chunk, sr=SR):
    mel = librosa.feature.melspectrogram(y=audio_chunk, sr=sr, n_fft=400, win_length=400, hop_length=HOP,
                                         n_mels=NUM_MELS, fmin=0, fmax=8000, power=2.0)
    db = librosa.power_to_db(mel, ref=np.max)
    padded = np.zeros((NUM_MELS, TIME_STEPS), dtype=np.float32)
    n = min(db.shape[1], TIME_STEPS)
    padded[:, :n] = db[:, :n]
    return torch.from_numpy(pad_aware_normalize(padded))[None, None]       # (1,1,80,T)

def chunk_audio(y, sr=SR, clip_seconds=CLIP_SECONDS, min_tail=5):
    n = clip_seconds * sr
    chunks = [y[i:i + n] for i in range(0, len(y), n)]
    if len(chunks) > 1 and len(chunks[-1]) < min_tail * sr:
        chunks = chunks[:-1]
    return chunks

@torch.no_grad()
def predict_chunks(model, y, sr=SR):
    """Sigmoid probabilities (n_chunks, 2) = [cut, dub]."""
    model.eval()
    dev = next(model.parameters()).device
    out = []
    for c in chunk_audio(y, sr):
        feats = build_features(mel_tensor(c, sr).to(dev), **FEATURE_CFG)
        out.append(torch.sigmoid(model(feats)).squeeze(0).cpu().numpy())
    return np.array(out)

def predict_path(model, path, sr=SR):
    """Clip-level probabilities for one file: max over chunks, per head."""
    y, _ = librosa.load(path, sr=sr, mono=True)
    return predict_chunks(model, y, sr).max(axis=0)

def classify_uploaded_media(model, sr=SR):
    if gfiles is None:
        raise RuntimeError("File upload needs Google Colab. Use predict_path(model, path) elsewhere.")
    print("Upload an audio (.mp3, .wav, .m4a) or video (.mp4, .mov, .avi) file:")
    uploaded = gfiles.upload()
    if not uploaded:
        print("No file uploaded.")
        return None
    filename = list(uploaded.keys())[0]
    try:
        y, _ = librosa.load(filename, sr=sr, mono=True)
    except Exception as e:
        print(f"Error reading '{filename}': {e}")
        return None

    total = len(y) / sr
    probs = predict_chunks(model, y, sr)
    classes = probs_to_class(probs)
    print(f"\n'{filename}': {total:.1f}s, {len(probs)} segment(s). Thresholds: {THRESHOLDS}")
    segments = []
    for i, (p, c) in enumerate(zip(probs, classes)):
        a, b = i * CLIP_SECONDS, min((i + 1) * CLIP_SECONDS, int(total))
        t = f"{a // 60:02d}:{a % 60:02d} - {b // 60:02d}:{b % 60:02d}"
        segments.append({'segment': i + 1, 'timestamp': t, 'label': CLASSES[c], 'p_cut': float(p[0]), 'p_dub': float(p[1])})
        print(f"Segment {i + 1:02d} [{t}] -> {CLASSES[c]:15s} p(cut)={p[0]:.2f} p(dub)={p[1]:.2f}")
    flags = (probs >= np.array([THRESHOLDS['cut'], THRESHOLDS['dub']])).any(axis=0)
    overall = _LUT[int(flags[0]), int(flags[1])]
    print("-" * 60)
    print(f"OVERALL (any segment flagged): {CLASSES[overall]} | max p(cut)={probs[:, 0].max():.2f} max p(dub)={probs[:, 1].max():.2f}")
    return {'filename': filename, 'overall_label': CLASSES[overall], 'segments': segments}
print("Inference utilities ready.")

Inference utilities ready.


### Preprocessing parity check

Raw training values near `-80 .. 0` indicate the `.npz` files were built with `power_to_db(ref=np.max)`, like inference. For a stronger check set `PARITY_AUDIO` (source audio of a training clip) and `PARITY_NPZ` (its `.npz` filename). The correlation of the normalized spectrograms over active frames should be above 0.99.

In [ ]:
PARITY_AUDIO = None   # e.g. "/content/source_clip.wav"
PARITY_NPZ = None     # e.g. "spectrogram_0001.npz"

name0 = labeled_records[0]['filename']
raw = np.load(NPZ_INDEX[name0]); raw = raw[raw.files[0]]
print(f"{name0}: shape={raw.shape} min={raw.min():.2f} max={raw.max():.2f} mean={raw.mean():.2f}")

if PARITY_AUDIO and PARITY_NPZ:
    y, _ = librosa.load(PARITY_AUDIO, sr=SR, mono=True)
    inf = mel_tensor(y[:CLIP_SECONDS * SR])[0, 0].numpy()
    tr = load_spec(NPZ_INDEX[PARITY_NPZ])
    n = int(min((np.abs(inf).sum(0) > 0).sum(), (np.abs(tr).sum(0) > 0).sum()))
    print(f"Parity correlation over {n} active frames: {np.corrcoef(inf[:, :n].ravel(), tr[:, :n].ravel())[0, 1]:.4f}")

AMIENRAISDIDUGAMELANGGGARHAMMenteriHakAsasiManusiaHAMNat7637066765763087637cut_001.npz: shape=(80, 3001) min=-44.23 max=35.77 mean=-11.39


## 11. Verification on Unseen Uploads

In [ ]:
results = classify_uploaded_media(model)

Upload an audio (.mp3, .wav, .m4a) or video (.mp4, .mov, .avi) file:
